In [ ]:
import os
import shutil
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)

# Kaggle paths
BASE_DIR = Path("/kaggle/input/datasets/asfakali2/iruvd-dataset-for-automatic-vehicle-detection/JU_yolov5")
OUTPUT_DIR = Path("/kaggle/working/iruvd_yolov5_processed")

# Source folders
VALID_IMAGES = BASE_DIR / "valid" / "images"
VALID_LABELS = BASE_DIR / "valid" / "labels"

# Destination folders
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train" / "labels").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "val" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "val" / "labels").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "test" / "images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "test" / "labels").mkdir(parents=True, exist_ok=True)

# Copy train set
print("Copying train set...")
for src_dir, dest_dir in [
    (BASE_DIR / "train" / "images", OUTPUT_DIR / "train" / "images"),
    (BASE_DIR / "train" / "labels", OUTPUT_DIR / "train" / "labels"),
]:
    for file in src_dir.iterdir():
        shutil.copy2(file, dest_dir / file.name)

# Split valid set into val (50%) and test (50%)
print("Splitting valid set into val/test...")
valid_images = list(VALID_IMAGES.glob("*.jpg")) + list(VALID_IMAGES.glob("*.png"))
random.shuffle(valid_images)

split_idx = int(len(valid_images) * 0.5)
val_files = valid_images[:split_idx]
test_files = valid_images[split_idx:]

def copy_split(files, split_name):
    for img_path in files:
        label_path = VALID_LABELS / img_path.with_suffix(".txt").name
        
        # Copy image
        shutil.copy2(img_path, OUTPUT_DIR / split_name / "images" / img_path.name)
        # Copy label if exists
        if label_path.exists():
            shutil.copy2(label_path, OUTPUT_DIR / split_name / "labels" / label_path.name)

copy_split(val_files, "val")
copy_split(test_files, "test")

print(f"Done! Split: {len(val_files)} val, {len(test_files)} test")

In [ ]:
yaml_content = """path: /kaggle/working/iruvd_yolov5_processed

train: train/images
val: val/images
test: test/images

nc: 14
names:
  0: truck
  1: cyclist
  2: bike
  3: tempo
  4: car
  5: jeep
  6: toto
  7: e-rickshaw
  8: auto-rickshaw
  9: bus
  10: van
  11: cycle-rickshaw
  12: person
  13: taxi
"""

with open("/kaggle/working/iruvd_yolov5_processed/data_corr.yaml", "w") as f:
    f.write(yaml_content)

print("data_corr.yaml saved!")

In [ ]:
from pathlib import Path

processed = Path("/kaggle/working/iruvd_yolov5_processed")
for split in ["train", "val", "test"]:
    imgs = list((processed / split / "images").glob("*.jpg"))
    labels = list((processed / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(labels)} labels")
    
import yaml
with open("/kaggle/working/iruvd_yolov5_processed/data_corr.yaml") as f:
    cfg = yaml.safe_load(f)
print(f"\nClasses: {cfg['nc']}")
print(f"Names: {cfg['names']}")

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()
print('\nultralytics', ultralytics.__version__)

In [ ]:
from pathlib import Path

EPOCHS   = 60
IMG_SIZE = 640
BATCH    = 8
MODEL    = 'rtdetr-l.pt'

DATA_YAML = Path('/kaggle/working/iruvd_yolov5_processed/data_corr.yaml')

N_PREVIEW    = 12
CONF         = 0.35

# Output directory
RUNS_DIR = Path('/kaggle/working/runs/road-obj-detection')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Config ready')
print(f'  Model   : {MODEL}')
print(f'  Epochs  : {EPOCHS} | Batch : {BATCH} | Img : {IMG_SIZE}')
print(f'  Data    : {DATA_YAML}')

In [ ]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print(f'\nClasses ({cfg["nc"]}) :')
for k, v in cfg["names"].items():
    print(f'  {k}: {v}')

In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import RTDETR

model = RTDETR(MODEL)

results = model.train(
    data = str(DATA_YAML),
    epochs = EPOCHS,
    imgsz = IMG_SIZE,
    batch = BATCH,
    device = [0,1],
    patience = 10,
    project = str(RUNS_DIR),
    name = 'rtdetr_train',
    exist_ok = True,

    # Optimization
    lr0 = 2e-4,
    weight_decay = 1e-4,
    warmup_epochs = 5,
    cos_lr = True,
    close_mosaic = 0,

    # Augmentations
    hsv_h = 0.015,
    hsv_s = 0.7,
    hsv_v = 0.4,
    flipud = 0.0,
    fliplr = 0.5,
    translate = 0.1,
    scale = 0.5,
)

BEST_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'

print('\nTraining complete')
print(f'Best weights → {BEST_WEIGHTS}')

In [ ]:
from IPython.display import Image, display

# Path to your training labels plot
labels_path = '/kaggle/working/runs/road-obj-detection/rtdetr_train/labels.jpg'

# Display the image
display(Image(filename=labels_path))

In [ ]:
import cv2, glob, math, time
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import RTDETR
import yaml
from pathlib import Path

BEST_WEIGHTS = "/kaggle/working/runs/road-obj-detection/rtdetr_train/weights/best.pt"
DATA_YAML = "/kaggle/working/iruvd_yolov5_processed/data_corr.yaml" 
RUNS_DIR = Path("/kaggle/working/runs/road-obj-detection")
VAL_DIR = Path("/kaggle/working/iruvd_yolov5_processed/train/images")

N_PREVIEW = 12
CONF = 0.45     
IMG_SIZE = 640  

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg["names"]

img_paths = sorted(glob.glob(str(VAL_DIR / "*.jpg")))[:N_PREVIEW]
if not img_paths:
    img_paths = sorted(glob.glob(str(VAL_DIR / "*.png")))[:N_PREVIEW]

if not img_paths:
    print(f'❌ No images found')
else:
    inf_model = RTDETR(BEST_WEIGHTS)
    preds = inf_model.predict(img_paths, conf=CONF, imgsz=IMG_SIZE, device=0, verbose=False)

    np.random.seed(42)
    COLORS = {name: tuple(np.random.randint(0, 255, 3).tolist()) for name in class_names.values()}

    frames = []
    for r in preds:
        # 1. Resize the image FIRST so drawing is relative to display size
        img = cv2.resize(r.orig_img, (640, 480)) 
        
        # Calculate scaling factors to adjust box coordinates
        h_orig, w_orig = r.orig_img.shape[:2]
        sw, sh = 640 / w_orig, 480 / h_orig

        if r.boxes:
            for box in r.boxes:
                # 2. Scale coordinates to the resized image
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                x1, y1, x2, y2 = int(x1 * sw), int(y1 * sh), int(x2 * sw), int(y2 * sh)
                
                cls_id = int(box.cls[0])
                cls_name = class_names[cls_id]
                conf_val = float(box.conf[0])
                color = COLORS[cls_name]

                # 3. Bolder Box (Thickness = 3)
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

                # 4. Bigger Text (fontScale = 0.6, thickness = 2)
                label = f'{cls_name} {conf_val:.2f}'
                (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)

                # Label background
                cv2.rectangle(img, (x1, y1 - th - 10), (x1 + tw, y1), color, -1)
                # White text for contrast
                cv2.putText(img, label, (x1, y1 - 7),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if frames:
        cols = min(4, len(frames))
        rows = math.ceil(len(frames) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4), facecolor='#111')
        
        fig.suptitle(f'RT-DETR on Test Set', fontsize=18, color='white', fontweight='bold', y=1.02)

        axes_list = [axes] if len(frames) == 1 else axes.flatten()
        for i, ax in enumerate(axes_list):
            if i < len(frames):
                ax.imshow(frames[i])
            ax.axis('off')

        plt.tight_layout()
        plt.show()